# 02. Tre approcci operativi, tutti gratuiti

Esistono diversi modi per ottenere un comportamento da decision model. Questo notebook ne confronta tre, tutti gratuiti o open:

1. Un modello open locale servito da Ollama o llama.cpp.
2. Una route ospitata gratuita, usata solo se non si ha hardware locale.
3. La lettura dei logit senza training, in cui un modello open valuta direttamente le opzioni dichiarate.

Caricheremo le configurazioni, esamineremo gli endpoint e confronteremo i client su input identici.


<div align='center'><img src='../images/backend_choices.png' alt='Tre backend gratuiti' style='width:80%; max-width:850px;'></div>

*Tutti e tre gli approcci restituiscono la stessa interfaccia: probabilità per le risposte dichiarate. Scegli in base a hardware e privacy, non al formato di output.*


## Passo 1. Leggere le tre configurazioni

Ogni file YAML descrive un backend. Li carichiamo uno alla volta e stampiamo un campo per cella, cosi e chiaro da dove arriva ogni valore.


In [ ]:
from decision_qa import load_config
local_ollama = load_config('configs/local_ollama.yaml')
print(local_ollama.backend)


In [ ]:
print(local_ollama.base_url)
print(local_ollama.endpoint)


In [ ]:
local_llamacpp = load_config('configs/local_llamacpp.yaml')
print(local_llamacpp.endpoint)


In [ ]:
hosted = load_config('configs/hosted.yaml')
print(hosted.backend)
print(hosted.model)


In [ ]:
assert local_ollama.endpoint.endswith('/v1/systemone')
assert hosted.endpoint.endswith('/v1/systemone')
print('all endpoints well formed')


## Passo 2. Il mock offline

Il mock non richiede server, chiave o rete. E lo strumento giusto mentre si imparano le strutture dati e mentre si eseguono i test. Lo chiamiamo con uno stato piccolo e due domande.


In [ ]:
from decision_qa import MockClient, NoulQuestion, ChoiceQuestion
state = {'context': 'The context says the treatment helped.', 'question': 'Did it help?'}
questions = [NoulQuestion(label='helped', statement='The treatment helped.', gold=True), ChoiceQuestion(label='verdict', options=['yes', 'no', 'unclear'], gold_index=0)]


In [ ]:
mock = MockClient(seed=0)
mock_results = mock.decision(state, questions)
print([r.question_label for r in mock_results])


In [ ]:
for r in mock_results:
    print(r.question_label, round(max(r.probabilities.values()), 3))


## Passo 3. Un client reale, solo se un server e attivo

Istanziamo un client reale ma non assumiamo che esista un server. Il blocco `try` tenta la chiamata e riporta un messaggio comprensibile se nessun modello locale e attivo. Questo mantiene il notebook eseguibile su una macchina appena configurata.


In [ ]:
from decision_qa import SystemOneClient
live_local = SystemOneClient(local_ollama)
print(live_local.config.endpoint)


In [ ]:
try:
    live_local.decision(state, questions)
except Exception as exc:
    print('local model not running:', type(exc).__name__)
else:
    print('local model answered')


## Passo 4. Lettura dei logit senza training

Metodi open come LitJev, SemIf e AnyJev trasformano qualunque modello open in un decision model senza fine-tuning. Il trucco e leggere la probabilita dei token delle etichette direttamente da un solo passaggio in avanti. Il nostro helper `question_labels` mostra quali etichette verrebbero lette per una data domanda.


In [ ]:
from decision_qa import question_labels
readout = question_labels(questions[1])
print(readout)


In [ ]:
assert readout == ['yes', 'no', 'unclear']
print('these are the option tokens to score')


## Passo 5. Confrontare la latenza su input identici

Misuriamo il tempo di ogni client disponibile sullo stesso stato e sulle stesse domande. Su questa macchina e disponibile solo il mock, quindi il confronto e un modello da completare dopo aver avviato un modello locale gratuito.


In [ ]:
import time
def timed(client, label):
    start = time.perf_counter()
    client.decision(state, questions)
    elapsed = (time.perf_counter() - start) * 1000
    print(label, round(elapsed, 1), 'ms')


In [ ]:
timed(mock, 'mock')


## Passo 6. Scegliere un approccio

Usa un modello open locale per privacy, lavoro offline e controllo totale. Usa una route ospitata gratuita solo quando non hai hardware locale e il piano gratuito basta. Usa la lettura dei logit per prototipi rapidi in cui non e accettabile fare training.

Implementazioni open da leggere: LitJev, SemIf, AnyJev e il supporto alle decisioni in llama.cpp documentato nella lista curata awesome-decision-models.
